# 条件概率与贝叶斯定理完整教程：从直觉到医学检验

## 📚 教学目标
1. 理解条件概率的定义：$P(B|A) = \frac{P(A \cap B)}{P(A)}$
2. 掌握乘法原理的条件概率形式
3. 理解全概率公式和贝叶斯定理
4. 手算贝叶斯定理的医学检验问题
5. 用模拟验证贝叶斯定理的结果

**参考书**：《基础统计学(第14版)》（Triola）第 4-3 节
**教学策略**：先用直觉理解条件概率，再学习贝叶斯公式，最后通过医学检验案例体会贝叶斯定理的实际意义

---

## 1. 场景设定：已知信息如何改变概率？

### 🎯 一个直觉问题

高尔夫球一杆进洞的概率为 1/12,500。但如果是**职业高尔夫球手**，该概率变为 1/2,500。

**核心问题**：当我们获得额外信息时，如何修正概率？

### 📖 书中的定义

> **条件概率**是指通过其他事件发生的额外信息所得的概率。
> 在事件 A 发生的条件下，事件 B 发生的概率，记作 $P(B|A)$。

### 📐 "至少一个" 的概率

本节第 1 部分先介绍一个实用技巧：

$$P(\text{至少一个}) = 1 - P(\text{一个都没有})$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 设置中文字体和绘图风格
import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. "至少一个" 的概率

### 📐 公式

$$P(\text{至少一个事件发生}) = 1 - P(\text{没有任何事件发生})$$

步骤：
1. 设 A 为至少有一个事件发生
2. $\bar{A}$ 为没有任何事件发生
3. 用乘法原理求 $P(\bar{A})$（通常比较简单）
4. $P(A) = 1 - P(\bar{A})$

In [ ]:
# ========== 步骤 1: "至少一个" - 产品缺陷率 ==========

# 书中例题: 工厂不良率 13%，购买 12 件，至少一件有缺陷的概率
p_defect = 0.13
n_items = 12

# 方法1: 对立事件
prob_no_defect = (1 - p_defect) ** n_items
prob_at_least_one = 1 - prob_no_defect

# 方法2: scipy.stats.binom 验证
# P(X>=1) = 1 - P(X=0)
prob_at_least_one_scipy = 1 - stats.binom.pmf(0, n_items, p_defect)

print("📊 步骤 1: '至少一个' 的概率 - 产品缺陷率")
print(f"  工厂不良率 = {p_defect}")
print(f"  购买数量 = {n_items}")
print(f"\n  方法1: 对立事件")
print(f"    P(无缺陷) = (1-{p_defect})^{n_items} = {prob_no_defect:.4f}")
print(f"    P(至少一件缺陷) = 1 - {prob_no_defect:.4f} = {prob_at_least_one:.4f}")
print(f"\n  方法2: scipy 验证")
print(f"    P(至少一件缺陷) = {prob_at_least_one_scipy:.4f}")
print(f"\n  ✅ 验证通过！")
print(f"\n💡 对立事件法大大简化了计算！")

In [ ]:
# ========== 可视化: 样本量与 "至少一个" 的概率 ==========

fig, ax = plt.subplots(figsize=(12, 6))

n_range = np.arange(1, 51)
prob_at_least_one = [1 - (1 - p_defect)**n for n in n_range]

ax.plot(n_range, prob_at_least_one, 'b-', linewidth=2, label=f'Defect Rate = {p_defect}')
ax.axhline(y=0.5, color='#e74c3c', linestyle='--', alpha=0.7, label='50% threshold')
ax.axhline(y=0.95, color='#e67e22', linestyle='--', alpha=0.7, label='95% threshold')
ax.fill_between(n_range, 0, prob_at_least_one, alpha=0.1, color='steelblue')

ax.set_xlabel('Number of Items Purchased', fontsize=12)
ax.set_ylabel('P(at least one defect)', fontsize=12)
ax.set_title('Probability of At Least One Defect vs Sample Size', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print(f"  不良率 = {p_defect*100:.0f}% 时：")
print(f"  - 购买 5 件: P(至少一件缺陷) ≈ {1-(1-p_defect)**5:.2f}")
print(f"  - 购买 12 件: P(至少一件缺陷) ≈ {1-(1-p_defect)**12:.2f}")
print(f"  - 购买 20 件: P(至少一件缺陷) ≈ {1-(1-p_defect)**20:.2f}")
print(f"  → 样本量越大，至少遇到一个缺陷的概率越高")

---

## 3. 条件概率

### 📐 条件概率公式

$$P(B|A) = \frac{P(A \cap B)}{P(A)}$$

其中：
- $P(B|A)$ = 在事件 A 已经发生的条件下，事件 B 发生的概率
- $P(A \cap B)$ = 事件 A 和事件 B 同时发生的概率
- $P(A)$ = 事件 A 发生的概率

> 💡 直觉方法：假设事件 A 已经发生，在缩小的样本空间中计算 B 的概率。

In [ ]:
# ========== 步骤 2: 条件概率 - 毒品检验 ==========

# 书中表 4-2: 毒品检验结果
#                  检验阳性    检验阴性    合计
# 吸食毒品           45          5         50
# 未吸食毒品         25         480       505
# 合计               70         485       555

total = 555
n_positive = 70
n_drugs = 50
n_positive_and_drugs = 45

# P(阳性|吸食毒品) - 直觉方法
# 假设受试者吸食毒品，那么他在 50 个吸食毒品的人中
# 其中 45 人检验阳性
prob_positive_given_drugs_intuition = n_positive_and_drugs / n_drugs

# P(阳性|吸食毒品) - 公式方法
prob_positive_and_drugs = n_positive_and_drugs / total
prob_drugs = n_drugs / total
prob_positive_given_drugs_formula = prob_positive_and_drugs / prob_drugs

# P(吸食毒品|阳性) - 直觉方法
# 假设检验结果为阳性，那么他在 70 个阳性的人中
# 其中 45 人吸食毒品
prob_drugs_given_positive_intuition = n_positive_and_drugs / n_positive

print("📊 步骤 2: 条件概率 - 毒品检验")
print(f"  总人数 = {total}")
print(f"  吸食毒品人数 = {n_drugs}")
print(f"  检验阳性人数 = {n_positive}")
print(f"  阳性且吸食毒品人数 = {n_positive_and_drugs}")
print(f"\n  P(阳性|吸食毒品):")
print(f"    直觉法: 45/50 = {prob_positive_given_drugs_intuition:.3f}")
print(f"    公式法: (45/555)/(50/555) = {prob_positive_given_drugs_formula:.3f}")
print(f"\n  P(吸食毒品|阳性):")
print(f"    直觉法: 45/70 = {prob_drugs_given_positive_intuition:.3f}")
print(f"\n  ⚠️ 条件概率悖论:")
print(f"    P(阳性|吸食毒品) = {prob_positive_given_drugs_intuition:.3f}")
print(f"    P(吸食毒品|阳性) = {prob_drugs_given_positive_intuition:.3f}")
print(f"    → 两者不相等！P(B|A) ≠ P(A|B)")

### 📐 条件概率悖论

广义上，$P(B|A) \neq P(A|B)$。

**经典例子**：
- D: 天黑了
- M: 午夜时分

$P(D|M) = 1$（午夜肯定天黑），但 $P(M|D) \approx 0$（天黑了不一定是午夜）。

> ⚠️ 在法律案件中，混淆被告有罪的概率与存在不利证据的概率，会导致严重的误判！

In [ ]:
# ========== 可视化: 条件概率 ==========

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图: P(阳性|吸食毒品) - 缩小到吸毒者群体
labels1 = ['Positive & Drug', 'Negative & Drug']
sizes1 = [45, 5]
colors1 = ['#e74c3c', '#2ecc71']
explode1 = (0.05, 0)

axes[0].pie(sizes1, explode=explode1, labels=labels1, colors=colors1,
            autopct='%1.1f%%', shadow=True, startangle=90, textprops={'fontsize': 12})
axes[0].set_title('P(Positive | Drug User) = 45/50 = 90.0%', fontsize=14, fontweight='bold')

# 右图: P(吸食毒品|阳性) - 缩小到阳性群体
labels2 = ['Drug & Positive', 'No Drug & Positive']
sizes2 = [45, 25]
colors2 = ['#e74c3c', '#3498db']
explode2 = (0.05, 0)

axes[1].pie(sizes2, explode=explode2, labels=labels2, colors=colors2,
            autopct='%1.1f%%', shadow=True, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('P(Drug User | Positive) = 45/70 = 64.3%', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  左图: 在 50 个吸毒者中，90% 检验阳性")
print("  右图: 在 70 个阳性者中，只有 64.3% 真正吸毒")
print("  → P(B|A) ≠ P(A|B)，条件概率是不对称的！")

---

## 4. 贝叶斯定理

### 📐 贝叶斯公式

$$P(A|B) = \frac{P(A) \times P(B|A)}{P(B)}$$

或展开为全概率公式形式：

$$P(A|B) = \frac{P(A) \times P(B|A)}{P(A) \times P(B|A) + P(\bar{A}) \times P(B|\bar{A})}$$

其中：
- $P(A)$ = 先验概率（获得新信息前的初始概率）
- $P(A|B)$ = 后验概率（获得新信息后的修正概率）
- $P(B|A)$ = 似然度

> 💡 贝叶斯定理的核心：根据新信息修正初始概率。

In [ ]:
# ========== 步骤 3: 贝叶斯定理 - 医学检验 ==========

# 书中例题: 癌症检验
# 发病率: P(C) = 0.01
# 假阳率: P(阳性|没有癌症) = 0.10
# 真阳率: P(阳性|有癌症) = 0.80

p_cancer = 0.01           # 先验概率
p_positive_given_cancer = 0.80    # 真阳率
p_positive_given_no_cancer = 0.10 # 假阳率
p_no_cancer = 1 - p_cancer

# 方法1: 构建假想总体（1000 人）
n_total = 1000
n_cancer = int(n_total * p_cancer)         # 10 人患癌
n_no_cancer = n_total - n_cancer            # 990 人未患癌
n_positive_cancer = int(n_cancer * p_positive_given_cancer)     # 8 人真阳性
n_negative_cancer = n_cancer - n_positive_cancer                # 2 人假阴性
n_positive_no_cancer = int(n_no_cancer * p_positive_given_no_cancer)  # 99 人假阳性
n_negative_no_cancer = n_no_cancer - n_positive_no_cancer             # 891 人真阴性
n_positive_total = n_positive_cancer + n_positive_no_cancer     # 107 人阳性总数

prob_cancer_given_positive_table = n_positive_cancer / n_positive_total

# 方法2: 贝叶斯公式
p_positive = (p_cancer * p_positive_given_cancer) + (p_no_cancer * p_positive_given_no_cancer)
prob_cancer_given_positive_bayes = (p_cancer * p_positive_given_cancer) / p_positive

print("📊 步骤 3: 贝叶斯定理 - 医学检验")
print(f"  先验概率: P(患癌) = {p_cancer}")
print(f"  真阳率: P(阳性|患癌) = {p_positive_given_cancer}")
print(f"  假阳率: P(阳性|未患癌) = {p_positive_given_no_cancer}")
print(f"\n  方法1: 构建假想总体 (1000 人)")
print(f"    患癌人数: {n_cancer}")
print(f"    未患癌人数: {n_no_cancer}")
print(f"    真阳性: {n_positive_cancer}")
print(f"    假阳性: {n_positive_no_cancer}")
print(f"    阳性总数: {n_positive_total}")
print(f"    P(患癌|阳性) = {n_positive_cancer}/{n_positive_total} = {prob_cancer_given_positive_table:.4f}")
print(f"\n  方法2: 贝叶斯公式")
print(f"    P(阳性) = P(C)×P(阳性|C) + P(Ā)×P(阳性|Ā)")
print(f"           = {p_cancer}×{p_positive_given_cancer} + {p_no_cancer}×{p_positive_given_no_cancer}")
print(f"           = {p_positive:.4f}")
print(f"    P(患癌|阳性) = (0.01 × 0.80) / {p_positive:.4f} = {prob_cancer_given_positive_bayes:.4f}")
print(f"\n  ✅ 两种方法结果一致！")
print(f"\n🎯 关键发现:")
print(f"  先验概率 P(患癌) = {p_cancer}")
print(f"  后验概率 P(患癌|阳性) = {prob_cancer_given_positive_bayes:.4f}")
print(f"  概率从 {p_cancer*100:.0f}% 增加到 {prob_cancer_given_positive_bayes*100:.1f}%")
print(f"  → 阳性结果不一定是坏消息，因为假阳率较高！")

In [ ]:
# ========== 可视化: 医学检验结果 ==========

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图: 1000 人的检验结果
labels = ['True Positive\n(8)', 'False Positive\n(99)', 'False Negative\n(2)', 'True Negative\n(891)']
sizes = [8, 99, 2, 891]
colors = ['#2ecc71', '#e74c3c', '#e67e22', '#3498db']
explode = (0.05, 0.05, 0, 0)

axes[0].pie(sizes, explode=explode, labels=labels, colors=colors,
            autopct='%1.1f%%', shadow=True, startangle=90, textprops={'fontsize': 10})
axes[0].set_title('Test Results for 1000 People', fontsize=14, fontweight='bold')

# 右图: 阳性结果中的构成
labels2 = ['True Cancer\n(8)', 'No Cancer\n(99)']
sizes2 = [8, 99]
colors2 = ['#e74c3c', '#2ecc71']

axes[1].pie(sizes2, labels=labels2, colors=colors2,
            autopct='%1.1f%%', shadow=True, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Among 107 Positive Results', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  左图: 1000 人的检验结果分布")
print("  右图: 在 107 个阳性结果中，只有 8 人真正患癌")
print("  → 大部分阳性结果是假阳性！")
print("  → 约 95% 的医生会将 P(患癌|阳性) 高估 10 倍")

---

## 5. 先验概率与后验概率

### 📐 定义

| 概念 | 定义 | 示例 |
|------|------|------|
| 先验概率 | 未获得额外信息时的初始概率 | P(患癌) = 0.01 |
| 后验概率 | 获得额外信息后的修正概率 | P(患癌|阳性) = 0.0748 |

> 💡 贝叶斯定理的核心价值：随着新信息的获取，不断修正概率估计。

In [ ]:
# ========== 步骤 4: 贝叶斯更新 - 多次检验 ==========

# 假设第一次检验阳性后，再做一次独立检验，结果也是阳性
# 第一次检验后: P(患癌|阳性) = 0.0748 (后验概率)
# 将后验概率作为第二次检验的先验概率

# 第一次检验
prior_1 = 0.01
posterior_1 = prob_cancer_given_positive_bayes

# 第二次检验（假设相同的真阳率和假阳率）
# 将第一次的后验作为第二次的先验
prior_2 = posterior_1
p_positive_2 = (prior_2 * p_positive_given_cancer) + ((1 - prior_2) * p_positive_given_no_cancer)
posterior_2 = (prior_2 * p_positive_given_cancer) / p_positive_2

# 第三次检验
prior_3 = posterior_2
p_positive_3 = (prior_3 * p_positive_given_cancer) + ((1 - prior_3) * p_positive_given_no_cancer)
posterior_3 = (prior_3 * p_positive_given_cancer) / p_positive_3

print("📊 步骤 4: 贝叶斯更新 - 多次检验")
print(f"  初始发病率: P(患癌) = {prior_1:.4f}")
print(f"  真阳率: {p_positive_given_cancer}")
print(f"  假阳率: {p_positive_given_no_cancer}")
print(f"\n  第一次检验阳性后:")
print(f"    先验: {prior_1:.4f}")
print(f"    后验: {posterior_1:.4f}")
print(f"\n  第二次检验也阳性后:")
print(f"    先验: {prior_2:.4f}")
print(f"    后验: {posterior_2:.4f}")
print(f"\n  第三次检验也阳性后:")
print(f"    先验: {prior_3:.4f}")
print(f"    后验: {posterior_3:.4f}")
print(f"\n💡 每次阳性结果都增加患癌的概率，这就是贝叶斯更新的力量！")

In [ ]:
# ========== 可视化: 贝叶斯更新过程 ==========

fig, ax = plt.subplots(figsize=(12, 6))

stages = ['Prior\n(Before Test)', 'After 1st\nPositive', 'After 2nd\nPositive', 'After 3rd\nPositive']
probs = [prior_1, posterior_1, posterior_2, posterior_3]
colors = ['#3498db', '#e67e22', '#e74c3c', '#c0392b']

bars = ax.bar(stages, probs, color=colors, edgecolor='white', alpha=0.8)

for bar, prob in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{prob:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('P(Cancer | Test Result)', fontsize=12)
ax.set_title('Bayesian Update: Probability of Cancer After Multiple Positive Tests',
             fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 0.5)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  蓝色: 先验概率 (1%)")
print("  橙色: 第一次阳性后 (7.48%)")
print("  红色: 第二次阳性后 (39.5%)")
print("  深红: 第三次阳性后 (83.8%)")
print("  → 每次新证据都更新我们的信念")

---

## 6. 大规模模拟验证

In [ ]:
# ========== 步骤 5: 模拟验证贝叶斯定理 ==========

n_simulations = 100000

# 模拟 1000 人的检验过程
# 1% 患癌
has_cancer = np.random.random(n_simulations) < p_cancer

# 检验结果
test_positive = np.zeros(n_simulations, dtype=bool)
for i in range(n_simulations):
    if has_cancer[i]:
        # 真阳率 80%
        test_positive[i] = np.random.random() < p_positive_given_cancer
    else:
        # 假阳率 10%
        test_positive[i] = np.random.random() < p_positive_given_no_cancer

# 计算 P(患癌|阳性)
positive_cases = test_positive.sum()
cancer_and_positive = (has_cancer & test_positive).sum()
prob_cancer_given_positive_sim = cancer_and_positive / positive_cases

# 理论值
prob_theory = prob_cancer_given_positive_bayes

print("📊 步骤 5: 模拟验证贝叶斯定理")
print(f"  模拟人数: {n_simulations:,}")
print(f"  患癌人数: {has_cancer.sum():,}")
print(f"  检验阳性人数: {positive_cases:,}")
print(f"  患癌且阳性人数: {cancer_and_positive:,}")
print(f"\n  模拟 P(患癌|阳性) = {prob_cancer_given_positive_sim:.4f}")
print(f"  理论 P(患癌|阳性) = {prob_theory:.4f}")
print(f"  偏差 = {abs(prob_cancer_given_positive_sim - prob_theory):.4f}")
print(f"\n  ✅ 模拟结果与贝叶斯定理一致！")

In [ ]:
# ========== 步骤 6: 不同先验概率下的后验概率 ==========

# 探索: 如果发病率不同，阳性结果的含义如何变化？

prior_probs = [0.001, 0.01, 0.05, 0.10, 0.20, 0.50]
posterior_probs = []

for prior in prior_probs:
    p_pos = (prior * p_positive_given_cancer) + ((1 - prior) * p_positive_given_no_cancer)
    posterior = (prior * p_positive_given_cancer) / p_pos
    posterior_probs.append(posterior)

print("📊 步骤 6: 先验概率对后验概率的影响")
print(f"  真阳率: {p_positive_given_cancer}")
print(f"  假阳率: {p_positive_given_no_cancer}")
print(f"\n  {'先验概率':>10}  {'后验概率':>10}  {'倍数变化':>10}")
print(f"  {'-'*10}  {'-'*10}  {'-'*10}")
for prior, posterior in zip(prior_probs, posterior_probs):
    ratio = posterior / prior
    print(f"  {prior:>10.3f}  {posterior:>10.4f}  {ratio:>10.1f}x")
print(f"\n💡 发病率越低，阳性结果的 '提升' 越大，但后验概率仍然可能很低")

In [ ]:
# ========== 可视化: 先验 vs 后验 ==========

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(prior_probs))
width = 0.35

bars1 = ax.bar(x - width/2, prior_probs, width, label='Prior Probability',
               color='#3498db', alpha=0.8, edgecolor='white')
bars2 = ax.bar(x + width/2, posterior_probs, width, label='Posterior Probability (After Positive Test)',
               color='#e74c3c', alpha=0.8, edgecolor='white')

ax.set_xlabel('Prior Probability of Disease', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Bayes Theorem: Prior vs Posterior Probability', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'{p:.3f}' for p in prior_probs])
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 图解说明：")
print("  蓝色: 先验概率 (发病率)")
print("  红色: 后验概率 (阳性检验后)")
print("  → 先验概率越低，阳性结果的 '信息量' 越大")
print("  → 但即使阳性，后验概率也可能不高（取决于假阳率）")

---

## 📌 核心概念回顾

### 📌 条件概率 (Conditional Probability)
- **公式**: $P(B|A) = \frac{P(A \cap B)}{P(A)}$
- **直觉**: 在 A 已发生的缩小样本空间中，B 的概率
- **关键**: $P(B|A) \neq P(A|B)$（条件概率悖论）

### 📌 贝叶斯定理 (Bayes' Theorem)
- **公式**: $P(A|B) = \frac{P(A) \times P(B|A)}{P(B)}$
- **全概率**: $P(B) = P(A) \times P(B|A) + P(\bar{A}) \times P(B|\bar{A})$
- **应用**: 根据新信息修正概率

### 📌 先验概率与后验概率
- **先验**: 获得新信息前的概率
- **后验**: 获得新信息后的概率
- **贝叶斯更新**: 后验成为下一次检验的先验

### 📌 "至少一个" 概率
- **公式**: $P(\text{至少一个}) = 1 - P(\text{一个都没有})$
- **技巧**: 用对立事件简化计算

### 🔗 完整流程
```
确定先验概率 → 获取新信息 → 计算似然度 → 应用贝叶斯公式 → 得到后验概率
     ↓              ↓           ↓              ↓              ↓
  P(患癌)=0.01   检验阳性    P(阳性|患癌)    贝叶斯公式    P(患癌|阳性)=0.075
```

---

## ❌ 常见误区

### ❌ 误区 1: 混淆 P(B|A) 和 P(A|B)
**✓ 正确理解**: 条件概率是不对称的！P(阳性|患癌) = 0.80，但 P(患癌|阳性) = 0.075。混淆两者是"检察官谬论"的根源。

### ❌ 误区 2: 忽略先验概率（基率忽略）
**✓ 正确理解**: 即使检验很准确（真阳率 80%），如果疾病本身罕见（1%），阳性结果的后验概率仍然很低。必须考虑基率（先验概率）。

### ❌ 误区 3: 将假阳率误认为是 "检验不准"
**✓ 正确理解**: 假阳率 10% 并不代表检验不准。在低发病率的情况下，大量健康人中的 10% 假阳性会淹没了少量真阳性。

### ❌ 误区 4: 认为一次阳性就能确诊
**✓ 正确理解**: 一次阳性检验只能将概率从 1% 提升到 7.5%，远未达到确诊标准。通常需要多次独立检验或更精确的确认检验。

### ❌ 误区 5: 忽略事件之间的独立性
**✓ 正确理解**: 贝叶斯更新假设每次检验是独立的。如果检验之间存在依赖关系（如同一实验室的系统误差），则不能简单地重复应用贝叶斯公式。